# Dense scientific ToT retrieval

Sentence Transformers supplies pretrained encoding; FAISS supplies indexing and search. The canonical adapter and the Phase 3 evaluator preserve corpus IDs and qrels. No model is fine-tuned.

From the repository root: `uv sync --locked --extra dense --extra notebook`, then `uv run python -m ipykernel install --sys-prefix --name scientific-tot-dense`. Select that kernel. CPU is the default; this notebook requires no Java. Set `TOT_DATA_ROOT` to restored data and `TOT_DENSE_OUTPUT` to a fresh result directory. Optionally set `TOT_BM25_BASELINE` to the matching representation's saved BM25 directory. See `docs/dense-retrieval.md` for substitution instructions and memory/truncation limits.

In [ ]:
import json
import os
from pathlib import Path

from scientific_tot import Dataset, DatasetAdapter
from scientific_tot.bibliography import title_abstract_from_markdown
from scientific_tot.integrations.dense import EncodingConfig, dense_experiment
from scientific_tot.integrations.pyterrier import select_demo_subset

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA = Path(
    os.environ.get(
        "TOT_DATA_ROOT", str(ROOT.parent / "multimodal-tip-of-tongue-for-scientific-papers/data")
    )
)
OUTPUT = Path(os.environ.get("TOT_DENSE_OUTPUT", str(ROOT / "results/dense-notebook")))
CACHE = Path(os.environ.get("TOT_DENSE_CACHE", str(ROOT / "results/dense-cache")))
BASELINE = os.environ.get("TOT_BM25_BASELINE")
REPRESENTATION = "fulltext"  # Or title_abstract with literal original sections.
DOCUMENTS, QUERIES = 250, 10  # Set both to None for all selected records.
CORPUS_SPLIT = "train+test"  # Select all for all available extractions.
config = EncodingConfig(device="cpu", batch_size=16)
config

In [ ]:
adapter = DatasetAdapter(Dataset(local_dir=DATA))
queries = adapter.load_queries(domain="ACL", split="test", mode="visual-and-text")
corpus = adapter.load_corpus(
    domain="ACL",
    split=CORPUS_SPLIT,
    representation="fulltext",
    missing="skip",
    include_metadata=False,
    include_figures=False,
)
corpus, queries = select_demo_subset(corpus, queries, DOCUMENTS, QUERIES)
if REPRESENTATION == "title_abstract":
    corpus = title_abstract_from_markdown(corpus, missing="error")
print(len(corpus), "documents", len(queries), "queries")

## Substitute a pretrained model

Replace `config` before running below. For example:
```python
config = EncodingConfig(
    model_id="sentence-transformers/all-MiniLM-L6-v2",
    model_revision="main",  # Resolved once to a commit SHA and recorded.
    profile="model", device="cpu", batch_size=16,
)
```
The default E5 model automatically receives `query: ` and `passage: ` through the library prompt API. Other models use `encode_query`/`encode_document` with saved prompts and task routing. When a model card requires instructions absent from its configuration, supply `query_prompt`, `document_prompt`, or their named prompt settings explicitly. E5 instruct variants require a task-specific query prompt. Match the model's cosine/dot setting. Long text is truncated by the model; no chunking is performed.

In [ ]:
metrics = dense_experiment(
    corpus, queries, OUTPUT / REPRESENTATION, cache=CACHE, config=config, baseline=BASELINE
)
metrics

In [ ]:
manifest = json.loads((OUTPUT / REPRESENTATION / "manifest.json").read_text())
print("Pinned model:", manifest["model"]["resolved_revision"])
print("Dataset:", manifest["dataset_revision_status"], manifest["corpus_sha256"])
print("Index:", manifest["index"])
print("Cache:", manifest["cache"])

In [ ]:
# Reuse embeddings and FAISS's persisted index in a fresh result directory.
cached_metrics = dense_experiment(
    corpus,
    queries,
    OUTPUT / (REPRESENTATION + "-cached"),
    cache=CACHE,
    config=config,
    baseline=BASELINE,
)
assert metrics.equals(cached_metrics)
assert (OUTPUT / REPRESENTATION / "run.trec").read_bytes() == (
    OUTPUT / (REPRESENTATION + "-cached") / "run.trec"
).read_bytes()
cached_manifest = json.loads((OUTPUT / (REPRESENTATION + "-cached") / "manifest.json").read_text())
assert cached_manifest["cache"]["document_hit"] and cached_manifest["cache"]["query_hit"]
cached_manifest["cache"]